# Model to Predict Number of Views for a TikTok Post

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split


In [3]:
df = pd.read_csv("../../processedDataset/preprocessedPostData.csv")

df.head()

,monthSin,monthCos,daySin,dayCos,hourSin,hourCos,weekdaySin,weekdayCos,hashtagVectors,nLikes,nShares,nFollowers,nComments,nViews,nAccountTotalLikes,postYear
0,-0.5,0.866025,0.988468,0.151428,-0.965926,-0.258819,-0.974928,-0.222521,[-0.06394912 -0.09934936 0.05018172 0.209499...,7.593374,1.386294,13.815512,2.944439,9.457279,15.363073,1.0
1,-0.5,0.866025,0.848644,0.528964,-0.258819,0.965926,0.433884,-0.900969,[-0.07862883 -0.10937244 0.03460374 0.202062...,8.905580,2.302585,13.815512,3.951244,10.874285,15.363073,1.0
2,-0.5,0.866025,0.724793,0.688967,-0.258819,-0.965926,0.974928,-0.222521,[-0.10907231 -0.05049138 0.07886021 0.172413...,8.439015,2.484907,13.815512,3.332205,10.537442,15.363073,1.0
3,-0.5,0.866025,0.571268,0.820763,-0.258819,0.965926,0.781831,0.623490,[-4.42794785e-02 -8.95778090e-02 4.70666662e-...,8.978660,1.945910,13.815512,3.218876,10.843514,15.363073,1.0
4,-0.5,0.866025,0.571268,0.820763,-0.258819,-0.965926,0.781831,0.623490,[-1.02420643e-01 -5.03370464e-02 7.44658336e-...,8.080237,2.302585,13.815512,2.708050,10.114599,15.363073,1.0


## Data Preparation

In [4]:
x = df[['nLikes', 'nShares', 'nFollowers']]
y = df['nViews']
xTrain, xTest, yTrain, yTest = train_test_split( x, y,test_size=0.2, random_state=100)

## Model Building

In [5]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, f1_score 

### Linear Regression

In [6]:
def trainModel(model, xTrain, xTest, yTrain, yTest, modelName):
    yTrainPred = model.predict(xTrain)
    yTestPred = model.predict(xTest)

    def calculateMetrics(y, yPred):
        mse = mean_squared_error(y, yPred)
        r2 = r2_score(y, yPred)
        rmse = mean_squared_error(y, yPred, squared=False)
        mae = mean_absolute_error(y, yPred)


        displayResults = pd.DataFrame([[modelName, mse, r2, rmse, mae]], columns=['Method', 'MSE', 'R2', 'RMSE', 'MAE'])
        print(displayResults)
        return displayResults

    print(f'--{modelName}--')
    
    print(f'Training')
    displayResultsTrain = calculateMetrics(yTrain, yTrainPred)

    print(f'Testing')
    displayResultsTest = calculateMetrics(yTest, yTestPred)

    displayResults = ''
    return displayResults

In [7]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(xTrain, yTrain)

results = trainModel(lr, xTrain, xTest, yTrain, yTest, "Linear Regression")
print(results)

--Linear Regression--
Training
              Method       MSE        R2      RMSE       MAE
0  Linear Regression  0.230154  0.960944  0.479743  0.370834
Testing
              Method      MSE        R2      RMSE       MAE
0  Linear Regression  0.22744  0.961353  0.476906  0.370788



In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'fit_intercept': [True, False],
    'normalize': [True, False]
}

lr = LinearRegression()

n_iter_search = 50

random_search = RandomizedSearchCV(lr, param_distributions=param_dist, n_iter=n_iter_search, cv=5)


random_search.fit(xTrain, yTrain)

print("Best parameters found: ", random_search.best_params_)


In [9]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LinearRegression

# Define your pipeline
pipeline = make_pipeline(StandardScaler(), LinearRegression())

# Define the hyperparameter space
param_dist = {
    'linearregression__fit_intercept': [True, False]
}

# Perform Random Search
random_search = RandomizedSearchCV(pipeline, param_distributions=param_dist, n_iter=10, cv=5)
random_search.fit(xTrain, yTrain)

# Print the best parameters
print("Best parameters found: ", random_search.best_params_)


/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/sklearn/model_selection/_search.py:296: UserWarning: The total space of parameters 2 is smaller than n_iter=10. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  UserWarning,


Best parameters found:  {'linearregression__fit_intercept': True}


In [10]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression(fit_intercept=True, normalize=True)
lr.fit(xTrain, yTrain)

results = trainModel(lr, xTrain, xTest, yTrain, yTest, "Linear Regression")
print(results)

--Linear Regression--
Training
              Method       MSE        R2      RMSE       MAE
0  Linear Regression  0.230154  0.960944  0.479743  0.370834
Testing
              Method      MSE        R2      RMSE       MAE
0  Linear Regression  0.22744  0.961353  0.476906  0.370788



/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/sklearn/linear_model/_base.py:145: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LinearRegression())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)


  FutureWarning,


In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RandomizedSearchCV

# Assuming xTrain and yTrain are your features and target variable
model = LinearRegression(fit_intercept=True, normalize=True)

# Define the hyperparameter space
param_dist = {
    'n_jobs': [-1, 1, 2, 3, 4, 5, 6]
}

# Create the random search model
random_search = RandomizedSearchCV(model, param_distributions=param_dist, n_iter=500, cv=7, random_state=42, n_jobs=-1)

# Fit the model
random_search.fit(xTrain, yTrain)

# Print the best parameters
print("Best Parameters: ", random_search.best_params_)

# Assuming trainModel is a function that trains the model and returns results
results = trainModel(random_search, xTrain, xTest, yTrain, yTest, "Linear Regression")
print(results)


/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/sklearn/model_selection/_search.py:296: UserWarning: The total space of parameters 7 is smaller than n_iter=500. Running 7 iterations. For exhaustive searches, use GridSearchCV.
  UserWarning,
/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/sklearn/linear_model/_base.py:145: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LinearRegression())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)


  FutureWarning,
/Library/Frameworks/Python.framework/

Best Parameters:  {'n_jobs': -1}
--Linear Regression--
Training
              Method       MSE        R2      RMSE       MAE
0  Linear Regression  0.230154  0.960944  0.479743  0.370834
Testing
              Method      MSE        R2      RMSE       MAE
0  Linear Regression  0.22744  0.961353  0.476906  0.370788



/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/sklearn/linear_model/_base.py:145: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the previous behavior:

from sklearn.pipeline import make_pipeline

model = make_pipeline(StandardScaler(with_mean=False), LinearRegression())

If you wish to pass a sample_weight parameter, you need to pass it as a fit parameter to each step of the pipeline as follows:

kwargs = {s[0] + '__sample_weight': sample_weight for s in model.steps}
model.fit(X, y, **kwargs)


  FutureWarning,
/Library/Frameworks/Python.framework/Versions/3.7/lib/python3.7/site-packages/sklearn/linear_model/_base.py:145: FutureWarning: 'normalize' was deprecated in version 1.0 and will be removed in 1.2.
If you wish to scale the data, use Pipeline with a StandardScaler in a preprocessing stage. To reproduce the p

## Random Forest

In [12]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(max_depth = 2, random_state=100)
rf.fit(xTrain, yTrain)

results = trainModel(rf, xTrain, xTest, yTrain, yTest, "Random Forest Regressor")
print(results)

--Random Forest Regressor--
Training
                    Method       MSE        R2      RMSE      MAE
0  Random Forest Regressor  0.783943  0.866968  0.885406  0.69523
Testing
                    Method       MSE        R2      RMSE       MAE
0  Random Forest Regressor  0.784067  0.866768  0.885476  0.689438



## Support Vector Regression (SVR)

In [14]:
from sklearn.svm import SVR

svr = SVR()
svr.fit(xTrain, yTrain)

results = trainModel(svr, xTrain, xTest, yTrain, yTest, "Support Vector Regression")
print(results)

--Support Vector Regression--
Training
                      Method       MSE        R2      RMSE       MAE
0  Support Vector Regression  0.224082  0.961974  0.473373  0.357101
Testing
                      Method       MSE        R2      RMSE       MAE
0  Support Vector Regression  0.219659  0.962675  0.468678  0.358352



## Gradient Boosting Regressor

In [15]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor()
gbr.fit(xTrain, yTrain)

results = trainModel(gbr, xTrain, xTest, yTrain, yTest, "Gradient Boosting Regressor")
print(results)

--Gradient Boosting Regressor--
Training
                        Method       MSE        R2      RMSE       MAE
0  Gradient Boosting Regressor  0.204877  0.965233  0.452633  0.349795
Testing
                        Method      MSE        R2      RMSE       MAE
0  Gradient Boosting Regressor  0.21001  0.964314  0.458268  0.356735



## XGBoost

In [17]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor()
xgb_model.fit(xTrain, yTrain)
results = trainModel(gbr, xTrain, xTest, yTrain, yTest, "XGB Regressor")
results

--XGB Regressor--
Training
          Method       MSE        R2      RMSE       MAE
0  XGB Regressor  0.204877  0.965233  0.452633  0.349795
Testing
          Method      MSE        R2      RMSE       MAE
0  XGB Regressor  0.21001  0.964314  0.458268  0.356735


''

## K-Nearest Neighbours

In [18]:
from sklearn.neighbors import KNeighborsRegressor

knn_model = KNeighborsRegressor(n_neighbors=5)
knn_model.fit(xTrain, yTrain)
results = trainModel(knn_model, xTrain, xTest, yTrain, yTest, "K Neighbors Regressor")
results 

predictions = knn_model.predict(xTest)
predictions


--K Neighbors Regressor--
Training
                  Method       MSE        R2      RMSE       MAE
0  K Neighbors Regressor  0.163683  0.972224  0.404578  0.308917
Testing
                  Method       MSE        R2    RMSE       MAE
0  K Neighbors Regressor  0.245619  0.958263  0.4956  0.380808


array([14.11635085, 11.04277525,  7.08205397, ..., 12.28658957,
       13.68439979, 11.00510386])

# Model for Time Created

In [21]:
X = df[['monthSin', 'monthCos', 'hourSin', 'hourCos', 'weekdaySin', 'weekdayCos']]
y = df['nShares', 'nComments']
xTrain, xTest, yTrain, yTest = train_test_split(X, y, test_size=0.2, random_state=42)


In [22]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(max_depth = 2, random_state=100)
rf.fit(xTrain, yTrain)

results = trainModel(rf, xTrain, xTest, yTrain, yTest, "Random Forest Regressor")
print(results)

--Random Forest Regressor--
Training
                    Method       MSE        R2     RMSE       MAE
0  Random Forest Regressor  5.170395  0.043722  2.27385  1.821442
Testing
                    Method       MSE        R2      RMSE       MAE
0  Random Forest Regressor  5.134233  0.044435  2.265885  1.830062



In [23]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(xTrain, yTrain)

results = trainModel(rf, xTrain, xTest, yTrain, yTest, "Random Forest Regressor")
print(results)

--Random Forest Regressor--
Training
                    Method       MSE        R2      RMSE       MAE
0  Random Forest Regressor  4.753945  0.120745  2.180354  1.735114
Testing
                    Method       MSE       R2      RMSE       MAE
0  Random Forest Regressor  5.230222  0.02657  2.286968  1.832596



## Retraining Models with Different Features

In [24]:
X = df[['monthSin', 'monthCos', 'hourSin', 'hourCos', 'weekdaySin', 'weekdayCos', 'nLikes']]
y = df['nComments']
xTrain, xTest, yTrain, yTest = train_test_split(X, y, test_size=0.2, random_state=42)

In [25]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(xTrain, yTrain)

results = trainModel(rf, xTrain, xTest, yTrain, yTest, "Random Forest Regressor")
print(results)

--Random Forest Regressor--
Training
                    Method       MSE        R2      RMSE       MAE
0  Random Forest Regressor  0.170534  0.968459  0.412958  0.281065
Testing
                    Method       MSE        R2     RMSE       MAE
0  Random Forest Regressor  1.154507  0.785127  1.07448  0.758068



In [27]:
import pandas as pd
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Feature and target definition
X = df[['monthSin', 'monthCos', 'hourSin', 'hourCos', 'weekdaySin', 'weekdayCos', 'nLikes']]
y = df['nComments']

# Initialize the model
rf = RandomForestRegressor(n_estimators=100, random_state=42)

# Define custom scoring functions
scoring = {
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred))),
    'MAE': make_scorer(mean_absolute_error),
    'R2': make_scorer(r2_score)
}

# Perform five-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {key: cross_val_score(rf, X, y, cv=kf, scoring=scoring[key]) for key in scoring}

# Calculate and print average metrics
for metric_name, scores in cv_results.items():
    print(f"Average {metric_name}: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

rf.fit(X, y)
y_pred = rf.predict(X)

overall_metrics = {
    'MSE': mean_squared_error(y, y_pred),
    'RMSE': np.sqrt(mean_squared_error(y, y_pred)),
    'MAE': mean_absolute_error(y, y_pred),
    'R2': r2_score(y, y_pred)
}

print("\nOverall Metrics on Full Dataset:")
for metric_name, value in overall_metrics.items():
    print(f"{metric_name}: {value:.4f}")


Average MSE: 1.1773 (+/- 0.0429)
Average RMSE: 1.0848 (+/- 0.0199)
Average MAE: 0.7513 (+/- 0.0039)
Average R2: 0.7820 (+/- 0.0073)

Overall Metrics on Full Dataset:
MSE: 0.1687
RMSE: 0.4107
MAE: 0.2825
R2: 0.9688


In [28]:
import pandas as pd
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Feature and target definition
X = df[['monthSin', 'monthCos', 'hourSin', 'hourCos', 'nComments', 'nLikes']]
y = df[['weekdaySin', 'weekdayCos']]

# Initialize the model
rf = RandomForestRegressor(n_estimators=100, random_state=42)

# Define custom scoring functions
scoring = {
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred))),
    'MAE': make_scorer(mean_absolute_error),
    'R2': make_scorer(r2_score)
}

# Perform five-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {key: cross_val_score(rf, X, y, cv=kf, scoring=scoring[key]) for key in scoring}

# Calculate and print average metrics
for metric_name, scores in cv_results.items():
    print(f"Average {metric_name}: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

rf.fit(X, y)
y_pred = rf.predict(X)

overall_metrics = {
    'MSE': mean_squared_error(y, y_pred),
    'RMSE': np.sqrt(mean_squared_error(y, y_pred)),
    'MAE': mean_absolute_error(y, y_pred),
    'R2': r2_score(y, y_pred)
}

print("\nOverall Metrics on Full Dataset:")
for metric_name, value in overall_metrics.items():
    print(f"{metric_name}: {value:.4f}")


Average MSE: 0.5457 (+/- 0.0019)
Average RMSE: 0.7387 (+/- 0.0013)
Average MAE: 0.6421 (+/- 0.0013)
Average R2: -0.0930 (+/- 0.0043)

Overall Metrics on Full Dataset:
MSE: 0.0763
RMSE: 0.2762
MAE: 0.2371
R2: 0.8472


In [ ]:
import numpy as np

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

rf = RandomForestRegressor(random_state=100)
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)
grid_search.fit(xTrain, yTrain)

# Best parameters from GridSearchCV
print(f"Best Parameters: {grid_search.best_params_}")

# Train the best model
best_rf = grid_search.best_estimator_
best_rf.fit(xTrain, yTrain)

# Evaluate the model
y_train_pred = best_rf.predict(xTrain)
y_test_pred = best_rf.predict(xTest)

# Calculate metrics
metrics = {
    'MSE': mean_squared_error,
    'RMSE': lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)),
    'MAE': mean_absolute_error,
    'R2': r2_score
}

# Print training and testing metrics
for name, metric in metrics.items():
    print(f"Training {name}: {metric(yTrain, y_train_pred)}")
    print(f"Testing {name}: {metric(yTest, y_test_pred)}")

# Feature importance
importances = best_rf.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False))